In [1]:
# Import Packages
import os
import re
import sys
import time
import copy
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd
from scipy import stats

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import r2_score, mean_squared_error

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, GroupKFold

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, RidgeCV

from sklearn.feature_selection import VarianceThreshold

#%load_ext autoreload
#%reload_ext autoreload
#%autoreload 2
#from model_functions import *

%run Model_functions.ipynb

In [2]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

!chmod 644 ~/.local/share/jupyter/history.sqlite
#!rm ~/.local/share/jupyter/history.sqlite

chmod: /Users/kosaraju_b/.local/share/jupyter/history.sqlite: No such file or directory


In [3]:
SENTINEL_DATA_CSV        = "../../DATA/AGB_DATA/Merged_Data/Sentinel_AGB/AGB_EO_SENTINEL.csv"
SENINEL_MISSING_DATA_CSV = "../../DATA/AGB_DATA/Merged_Data/Sentinel_AGB/AGB_VAL_EO_SENTINEL.csv"

# SENTINEL DATA

In [4]:
sentinel_df = pd.read_csv(SENTINEL_DATA_CSV)
print(sentinel_df.shape)
sentinel_df.columns

(8774, 24)


Index(['dataset', 'plot_id', 'start_date', 'end_date', 'latitude_x',
       'longitude_x', 'diameter', 'height', 'species', 'plant_AGB_kg',
       'longitude_y', 'latitude_y', 'time', 'Blue', 'Green', 'Red', 'NIR',
       'SWIR1', 'SWIR2', 'NDVI', 'MNDWI', 'NBR', 'EVI', 'CLOUD_COVERAGE'],
      dtype='object')

In [5]:
sentinel_df = sentinel_df.drop(columns=['latitude_y', 'longitude_y'])
sentinel_df.columns

Index(['dataset', 'plot_id', 'start_date', 'end_date', 'latitude_x',
       'longitude_x', 'diameter', 'height', 'species', 'plant_AGB_kg', 'time',
       'Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2', 'NDVI', 'MNDWI', 'NBR',
       'EVI', 'CLOUD_COVERAGE'],
      dtype='object')

## DATA PREPROCESSING

In [6]:
#sentinel_df = handle_null_data(sentinel_df)
sentinel_df = sentinel_df.dropna()
sentinel_df.columns

Index(['dataset', 'plot_id', 'start_date', 'end_date', 'latitude_x',
       'longitude_x', 'diameter', 'height', 'species', 'plant_AGB_kg', 'time',
       'Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2', 'NDVI', 'MNDWI', 'NBR',
       'EVI', 'CLOUD_COVERAGE'],
      dtype='object')

In [7]:
non_feature_cols = [
    'plant_AGB_kg',        # Target variable
    #'dataset',             # metadata
    'start_date',          # metadata
    'end_date',            # metadata
    'capture_start',       # metadata
    'capture_end',         # metadata
    #'latitude',            # coordinate
    #'longitude',           # coordinate
]
sentinel_bands = [
    'Blue', 'Green', 'Red'
]
sentinel_indices = [
    'NIR', 'SWIR1', 'SWIR2', 'NDVI',
    'MNDWI', 'NBR', 'EVI', 'CLOUD_COVERAGE'
]

target = 'plant_AGB_kg'
feature_cols = [c for c in sentinel_df.columns if c not in non_feature_cols]

### Find correlations

Columns with a Pearson correlation coefficient (r) between -0.1 and 0.1 with the target variable are generally considered to have a negligible or very weak relationship.

### Remove Low Variance Features (cols)

### Remove Features With Weak Correlation to Target

### Convert categorical variables to one-hot encoding

# LINEAR REGRESSION

In [8]:
%run Model_functions.ipynb

# RANDOM FOREST

In [9]:
sentinel_df['dataset'].unique()

array(['ElSalvador', 'CostaRica-Nicoya', 'Belige', 'Honduras-Blanca',
       'Brazil-Manguezal', 'Brazil-Maruipe', 'Brazil-AcarauBoca',
       'Brazil-Barreto', 'Brazil-Salinas', 'Brazil-Caetano'], dtype=object)

In [ ]:
datasets = {}

for dataset_name in sentinel_df['dataset'].unique():
    datasets[dataset_name] = sentinel_df[sentinel_df['dataset'] == dataset_name].drop(columns=['dataset']).copy()
    print(f"{dataset_name:30s} : {len(datasets[dataset_name])} rows")

In [ ]:
brazil_real    = ['Brazil-FuroGrande', 'Brazil-BocaGrande', 
                  'Brazil-Furo_Do_Chato', 'Brazil-Mangue',
                  'Brazil-Caetano', 'Brazil-Maruipe',
                  'Brazil-Barreto', 'Brazil-Salinas']

brazil_snapped = ['Brazil-AcarauBoca', 'Brazil-Manginho',
                  'Brazil-Manguezal', 'Brazil-Rego']

In [ ]:
brazil_real_df = pd.concat(
    [datasets[k] for k in brazil_real if k in datasets],
    axis=0
).reset_index(drop=True)

print(f"Brazil real datasets merged : {[k for k in brazil_real if k in datasets]}")
print(f"Total rows                  : {len(brazil_real_df)}")

# Update datasets dictionary
datasets['Brazil-Real'] = brazil_real_df

# Remove individual entries
for key in brazil_real:
    if key in datasets:
        del datasets[key]

print(f"\nUpdated dataset keys: {list(datasets.keys())}")

In [ ]:
non_feature_cols = [
    'plant_AGB_kg',        # Target variable
    #'dataset',             # metadata
    'start_date',          # metadata
    'end_date',            # metadata
    'capture_start',       # metadata
    'capture_end',         # metadata
    'latitude',            # coordinate
    'longitude',           # coordinate
]

sentinel_bands = [
    'Blue', 'Green', 'Red'
]
sentinel_indices = [
    'NIR', 'SWIR1', 'SWIR2', 'NDVI',
    'MNDWI', 'NBR', 'EVI', 'CLOUD_COVERAGE'
]
useful_categorical = ['plot_id', 'species']

target = 'plant_AGB_kg'
feature_cols = [c for c in sentinel_df.columns if c not in non_feature_cols]
feature_cols

In [ ]:
categorical_cols_orig = get_categorical_cols(sentinel_df)
print("categorical_cols: %s" %categorical_cols_orig)

In [ ]:
for dataset_name in datasets.keys():
    if dataset_name == 'Brazil':
        continue

    print(f"\n{dataset_name}")
    data_df = datasets[dataset_name]

    #label = f"Region: {dataset_name}. Features: Sentinel bands only"
    #X = data_df[useful_categorical + sentinel_bands]

    #label = f"Region: {dataset_name}. Features: Sentinel Indices only"
    #X = data_df[useful_categorical + sentinel_indices]

    #label = f"Region: {dataset_name}. Features: Sentinel bands + Indices"
    #X = data_df[useful_categorical + sentinel_bands + sentinel_indices]
    
    label = f"Region: {dataset_name}. Features: Height alone"
    X = data_df[['height', 'plot_id']]

    y = data_df[target]

    groups = X['plot_id'].copy()
    X = X.drop(columns=['plot_id'])
    #print(f"X.shape: {X.shape}")
    print(f"Features: {X.columns}")
    #print(f"y.shape: {y.shape}")

    categorical_cols = get_categorical_cols(X)
    #print("categorical_cols: %s" %categorical_cols)
    X = pd.get_dummies(X, columns=categorical_cols, dtype=int)
    #print(f"Features with dummies: {X.columns}")
    
    results = randomForest_groups(X,
                                  y,
                                  groups,
                                  label)
    show_importances(results)